In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# load the dataset
df = pd.read_csv('./data/raw/Dataset.csv')

# missing values with empty strings to prevent errors during text concatenation
features_to_combine = ['type', 'director', 'country', 'rating', 'listed_in']
for feature in features_to_combine:
    df[feature] = df[feature].fillna('')

# a combined feature string for each title
def create_metadata_soup(row):
    return f"{row['type']} {row['director']} {row['country']} {row['rating']} {row['listed_in']}"

df['combined_features'] = df.apply(create_metadata_soup, axis=1)

# Display the first few combined features
df[['title', 'combined_features']].head()


,title,combined_features
0,Dick Johnson Is Dead,Movie Kirsten Johnson United States PG-13 Docu...
1,Ganglands,TV Show Julien Leclercq France TV-MA Crime TV ...
2,Midnight Mass,TV Show Mike Flanagan United States TV-MA TV D...
3,Confessions of an Invisible Girl,Movie Bruno Garotti Brazil TV-PG Children & Fa...
4,Sankofa,"Movie Haile Gerima United States TV-MA Dramas,..."


In [3]:
# Initialize the TF-IDF Vectorizer
# Removing standard English stop words to reduce noise
tfidf = TfidfVectorizer(stop_words='english')

# Fit and transform the combined features into a numerical matrix
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")


TF-IDF Matrix Shape: (8790, 6569)


In [5]:
# Compute the cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a reverse mapping of titles to dataset indices for quick lookups
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

print("Similarity matrix calculated successfully.")


Similarity matrix calculated successfully.


In [ ]:
def get_recommendations(title, cosine_sim=cosine_sim, num_recommendations=5):
    #if the title exists in the dataset
    if title not in indices:
        return f"Title '{title}' not found in the dataset."
    
    #the index of the title
    idx = indices[title]
    
    # pairwise similarity scores of all titles with that title
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # the titles based on similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # the scores of the top N most similar titles (excluding itself at index 0)
    sim_scores = sim_scores[1:num_recommendations+1]
    
    # the title indices
    movie_indices = [i[0] for i in sim_scores]
    
    # most similar titles with their similarity scores
    recommendations = df.iloc[movie_indices][['title', 'listed_in', 'director']].copy()
    recommendations['similarity_score'] = [round(i[1], 4) for i in sim_scores]
    
    return recommendations

# the recommendation engine
get_recommendations('Inception', num_recommendations=2)


,title,listed_in,director,similarity_score
8019,Star Trek: The Next Generation,"TV Action & Adventure, TV Sci-Fi & Fantasy",Not Given,0.5171
8105,Star Trek: Voyager,"TV Action & Adventure, TV Sci-Fi & Fantasy",Not Given,0.5171


In [ ]:
def evaluate_recommendation_quality(target_title, recommended_df):
    if target_title not in indices:
        return "Cannot evaluate."
        
    target_genres = set(df.loc[indices[target_title], 'listed_in'].split(', '))
    
    overlap_scores = []
    for genres in recommended_df['listed_in']:
        rec_genres = set(genres.split(', '))
        # Calculate Jaccard Similarity for genres: (Intersection / Union)
        intersection = len(target_genres.intersection(rec_genres))
        union = len(target_genres.union(rec_genres))
        overlap_scores.append(intersection / union if union != 0 else 0)
    
    avg_overlap = np.mean(overlap_scores) * 100
    
    print(f"Evaluation for '{target_title}':")
    print(f"Average Genre Overlap (Jaccard Index): {avg_overlap:.1f}%")
    print("Higher percentages indicate stronger category alignment between the target and recommendations.")
    return avg_overlap

#recommendations and evaluate them
target = 'Inception'
recs = get_recommendations(target, num_recommendations=2)
evaluate_recommendation_quality(target, recs)


Evaluation for 'Inception':
Average Genre Overlap (Jaccard Index): 0.0%
Higher percentages indicate stronger category alignment between the target and recommendations.


np.float64(0.0)